# IBGE Municipality Population - Silver Transformation

## Parameters

In [0]:
dbutils.widgets.text(
    name="environment",
    defaultValue="dev",
    label="Environment"
)

environment = dbutils.widgets.get("environment").strip().lower()

if environment not in ("dev", "test", "prod"):
    raise ValueError(
        f"Unsupported environment: {environment}. Expected dev, test, or prod."
    )

## Setup

In [0]:
from pyspark.sql.functions import col, split, lower, translate

catalog = f"ecommerce_{environment}"
source_table = f"{catalog}.bronze.ibge_municipality_population"
target_table = f"{catalog}.silver.ibge_municipality_population"

## Read Bronze data

In [0]:
bronze_df = spark.table(source_table)

## Transform to Silver

In [0]:
silver_df = bronze_df.withColumnRenamed("municipality_name", "municipality_name_raw")

In [0]:
silver_df = (
    silver_df
    .withColumn(
        "municipality_name",
        split(col("municipality_name_raw"), ' \\(')[0]
        )
    .withColumn(
        "state_code",
        split(split(col("municipality_name_raw"), ' \\(')[1], '\\)')[0]
    )
)

In [0]:
silver_df = silver_df.withColumn(
    "municipality_name",
    translate(
        lower(col("municipality_name")),
        "áàâãäéèêëíìîïóòôõöúùûüç-'",
        "aaaaaeeeeiiiiooooouuuuc  "
    )
)

In [0]:
silver_df = silver_df.withColumn("year", col("year").cast("int"))

silver_df = (
    silver_df
    .filter(
        col("population")
        .rlike("^[0-9]+$")
        )
    .withColumn(
        "population",
        col("population").cast("int"))
)

## Write to Silver

In [0]:
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)